DREAMwalk - split `{random, disease_area}` x embedding graph `{no leakage, leakage}`. this is the leakage ablation: the only difference between the two embedding graphs is whether MSI edge type 6 (`6_drug_indication_df.tsv`) was there when the node embeddings were trained. everything else is thesame.

In [118]:
import json, glob, os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats

# results csvs are present in `msi_cv/` and `msi_cv_ablation/`. `cv_metrics.csv` has split on random and `cv_metrics_disease_area.csv` has split as disease area.  

DREAMWALK_ROOT = Path('/home/sid/Forge/DREAMwalk')
KGDR_ROOT = Path('/home/sid/Forge/knowledge-graphs-drug-repurposing')
KGDR_RESULTS_DIR  = KGDR_ROOT / 'results/oregano/classifier_results'

# the leakage ablation is a binary identity encoding; split mode is carried by facets
GRAPH_ORDER = ['no leakage', 'leakage']
SPLIT_ORDER = ['random', 'disease_area']
METRICS = ['auroc', 'aupr']
METRIC_LABEL = {'auroc': 'AUROC', 'aupr': 'AUPRC'}

## split vs leakage results in a table

`msi_cv/` = embeddings trained on the graph without drug-disease edges ("fixed").

`msi_cv_ablation/` = same as the paper (should have called it something other than "ablation" coz it's not really that)

In [119]:
CONDITIONS = [ # leakage, split, path to metrics
    ('no leakage', 'random', 'msi_cv/cv_metrics_random.csv'),
    ('no leakage', 'disease_area', 'msi_cv/cv_metrics_disease_area.csv'),
    ('leakage', 'random', 'msi_cv_ablation/cv_metrics_random.csv'),
    ('leakage', 'disease_area', 'msi_cv_ablation/cv_metrics_disease_area.csv'),
]

frames = []
for graph, split, rel in CONDITIONS:
    d = pd.read_csv(DREAMWALK_ROOT / rel)
    d.insert(0, 'graph', graph)
    d['split_mode'] = split
    frames.append(d)
dw = pd.concat(frames, ignore_index=True)
dw['graph'] = pd.Categorical(dw['graph'], GRAPH_ORDER, ordered=True)
dw['split_mode'] = pd.Categorical(dw['split_mode'], SPLIT_ORDER, ordered=True)

# `msi_cv/cv_metrics.csv` is an earlier partial random run (6 seeds) superseded by cv_metrics_random.csv (10 seeds) and is left out
print(dw.groupby(['graph', 'split_mode'], observed=True)
        .agg(splits=('aupr', 'size'), 
             seeds=('seed', 'nunique'),
             folds_per_seed=('fold', 'nunique'), 
             n_test_med=('n_test', 'median'), 
             pos_rate=('test_pos_rate', 'mean')).round(3))

                         splits  seeds  folds_per_seed  n_test_med  pos_rate
graph      split_mode                                                       
no leakage random            50     10               5      2370.0      0.50
           disease_area     100     10              10      1180.0      0.53
leakage    random            50     10               5      2370.0      0.50
           disease_area     100     10              10      1180.0      0.53


### metrics

In [124]:
seed_lvl = (dw.groupby(['graph', 'split_mode', 'seed'], observed=True)[METRICS].mean().reset_index())

def fmt_cell(g):
    return pd.Series({METRIC_LABEL[m]: f'{g[m].mean():.3f}, std = {g[m].std():.3f}' for m in METRICS})

table1 = (seed_lvl.groupby(['graph', 'split_mode'], observed=True)
                  .apply(fmt_cell, include_groups=False))
table1.index.names = ['embedding graph', 'split']
print(table1)

# the same table in numeric
table1_num = (seed_lvl.groupby(['graph', 'split_mode'], observed=True)[METRICS].agg(['mean', 'std']).round(4))

                                           AUROC               AUPRC
embedding graph split                                               
no leakage      random        0.915, std = 0.003  0.922, std = 0.002
                disease_area  0.663, std = 0.033  0.701, std = 0.028
leakage         random        0.980, std = 0.001  0.979, std = 0.001
                disease_area  0.933, std = 0.009  0.935, std = 0.009


### check variance in different runs, 10 seeds

In [121]:
def paired(long, index_col, a, b, label_col, label_fmt):
    out = []
    # pivot table on index_col and seed, with label_col as columns, and METRICS as values
    piv = long.pivot_table(index=[index_col, 'seed'], columns=label_col, values=METRICS, observed=True)
    for key, sub in piv.groupby(level=0):
        row = {label_fmt: key, 'num_seeds': len(sub)}
        for m in METRICS:
            d = (sub[(m, a)] - sub[(m, b)]).dropna()
            row[f'delta {METRIC_LABEL[m]}'] = f'mean = {d.mean():.3f}, std = {d.std():.3f}'
        out.append(row)
    return pd.DataFrame(out)

leak = paired(seed_lvl, 'split_mode', 'leakage', 'no leakage', 'graph', 'split')
leak.insert(0, 'effect', 'leakage inflation')
hard = paired(seed_lvl, 'graph', 'random', 'disease_area', 'split_mode', 'split')
hard = hard.rename(columns={'split': 'embedding graph'})
hard.insert(0, 'effect', 'split hardness')

table2 = pd.concat([leak, hard], ignore_index=True)
table2 = table2[['effect', 'split', 'embedding graph'] + [c for c in table2.columns if c.startswith(('delta'))]]
print(table2.fillna('').set_index(['effect', 'split', 'embedding graph']))

paper_like = seed_lvl.query('graph == "leakage" and split_mode == "random"')['aupr']
honest = seed_lvl.query('graph == "no leakage" and split_mode == "disease_area"')['aupr']

print(f"leakage random mean AUPR = {paper_like.mean():.3f}, std = {paper_like.std():.3f}")
print(f"honest random mean AUPR = {honest.mean():.3f}, std = {honest.std():.3f}")

                                                              delta AUROC  \
effect            split        embedding graph                              
leakage inflation random                        mean = 0.065, std = 0.002   
                  disease_area                  mean = 0.270, std = 0.027   
split hardness                 no leakage       mean = 0.252, std = 0.033   
                               leakage          mean = 0.046, std = 0.008   

                                                              delta AUPRC  
effect            split        embedding graph                             
leakage inflation random                        mean = 0.057, std = 0.001  
                  disease_area                  mean = 0.233, std = 0.021  
split hardness                 no leakage       mean = 0.221, std = 0.027  
                               leakage          mean = 0.045, std = 0.009  
leakage random mean AUPR = 0.979, std = 0.001
honest random mean AUPR = 0.701, st

/tmp/ipykernel_77502/791153113.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for key, sub in piv.groupby(level=0):
/tmp/ipykernel_77502/791153113.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for key, sub in piv.groupby(level=0):


## cross project metrics

DREAMwalk has a 1:1 positive:negative set, we have a ~1:10 set. but AUPRC depends on exact imbalance (prevalance, which is how big the positive class is!)

TODO could also use precision recall gain curves https://papers.nips.cc/paper_files/paper/2015/file/33e8075e9970de0cfea955afd4644bb2-Paper.pdf

So for now do:
$$\text{normalised AUPRC} = \frac{\text{AUPRC} - \text{prevalence}}{1 - \text{prevalence}}$$

which is a linear scaling, but at least is more comparable!

## KGDR model selection

need to only look at ComplEx models since they were the best performing, and ignore TransE since there was clear data leakage in the way the embedding space was stratified. best AUPRC was ~0.85 with svm_poly and complex. mlp_small and mlp with DistMult/ComplEx was also good.

In [122]:
# KGDR, cv summaries and holdout results per split regime
kg_cv, kg_hold = [], []
for f in sorted(KGDR_RESULTS_DIR.glob('kfold_cv_summary_*.csv')):
    d = pd.read_csv(f)
    # get regime name from file stem, for eg., 'kfold_cv_summary_random' -> 'random'
    d['regime'] = f.stem[len('kfold_cv_summary_'):]
    kg_cv.append(d)
    # print(d)
for f in sorted(KGDR_RESULTS_DIR.glob('holdout_results_*.csv')):
    d = pd.read_csv(f)
    # get regime name from file stem, for eg., 'holdout_results_random' -> 'random'
    d['regime'] = f.stem[len('holdout_results_'):]
    kg_hold.append(d)
    # print(d)
kg_cv, kg_hold = pd.concat(kg_cv, ignore_index=True), pd.concat(kg_hold, ignore_index=True)

# drop all transe rows since there was data leakage in the way embedding space was stratified. and split mode should only be emb_space_diverse, nothing else as that's what we should focus on
kg_cv = kg_cv[kg_cv['kge'] != 'transe']
kg_hold = kg_hold[kg_hold['kge'] != 'transe']

kg_cv = kg_cv[kg_cv['regime'] == 'emb_space_diverse']
kg_hold = kg_hold[kg_hold['regime'] == 'emb_space_diverse']

# sort kg_cv by regime, model, kge, then by auprc_mean descending
kg_cv = kg_cv.sort_values(['auprc_mean'], ascending=[False])
# print(kg_cv[['model', 'kge', 'auroc_mean', 'auprc_mean']])

kg_hold = kg_hold.sort_values(['AUPRC'], ascending=[False])
# print(kg_hold[['model', 'kge', 'AUROC', 'AUPRC']])

# holdout prevalence, from the confusion matrices in the json sibling files
hold_prev = {}
for f in sorted(KGDR_RESULTS_DIR.glob('holdout_results_*.json')):
    d = json.load(open(f))
    for kges in d.values():
        for v in kges.values():
            cm = v.get('confusion_matrix')
            if cm:
                hold_prev[f.stem[len('holdout_results_'):]] = (cm['tp'] + cm['fn']) / sum(cm.values())
                break
        break
prevalance_KGDR = float(np.mean(list(hold_prev.values()))) # the holdout set is the same across regimes
print('KGDR prevalence in holdout per regime:', {k: round(v, 4) for k, v in hold_prev.items()})

# cv - select the cell, report its holdout score
sel = kg_cv.sort_values('auprc_mean', ascending=False).groupby('regime', as_index=False).head(1)
sel = sel[['regime', 'model', 'kge', 'auroc_mean', 'auprc_mean']]
kg_rows = sel.merge(kg_hold[['regime', 'model', 'kge', 'AUROC', 'AUPRC']], on=['regime', 'model', 'kge'])
kg_rows['prevalence'] = kg_rows['regime'].map(hold_prev).fillna(prevalance_KGDR)
kg_rows[['regime', 'model', 'kge', 'auroc_mean', 'auprc_mean', 'AUROC', 'AUPRC', 'prevalence']]
print(kg_rows)

KGDR prevalence in holdout per regime: {'emb_space_diverse': 0.0918, 'hard': 0.0918, 'non_strat_random': 0.0884, 'seq_max_dist': 0.0918}
              regime     model      kge  auroc_mean  auprc_mean     AUROC  \
0  emb_space_diverse  svm_poly  complex    0.918442    0.803808  0.983389   

      AUPRC  prevalence  
0  0.850073    0.091764  


In [123]:
"""
   regime             model     kge        auroc_mean  auprc_mean     AUROC      AUPRC       prevalence
   emb_space_diverse  svm_poly  complex    0.918442    0.803808       0.983389   0.850073    0.091764
"""
def norm_auprc(a, prevalence):
    return (a - prevalence) / (1 - prevalence)

rows = []
# for dreamwalk
for (g, s), sub in seed_lvl.groupby(['graph', 'split_mode'], observed=True):
    # prevalence is per-split, not carried on seed_lvl, so take it from the fold-level frame
    # eval is 10 seeds, 10 repeats for disease_area, 5 folds for random
    prevalence = dw.query('graph == @g and split_mode == @s')['test_pos_rate'].mean()
    rows.append(dict(project='DREAMwalk', setting=f'{s} / {g}',
                     what='MeSH-category holdout' if s == 'disease_area' else 'random pairs',
                     model='XGBoost',
                     prevalence=prevalence, 
                     AUROC=sub['auroc'].mean(),
                     AUPRC=sub['aupr'].mean(),
                     AUROC_std=sub['auroc'].std(), 
                     AUPRC_std=sub['aupr'].std()))

# for kgdr
for _, r in kg_rows.iterrows():
    # first add the holdout row, then the CV row
    # eval is fixed holdout size for the holdout row, and 10 different folds for the CV row
    rows.append(dict(project='KGDR', 
                     setting=r['regime'], 
                     what="embedding space diverse",
                     model=f"{r['model']} / {r['kge']}",
                     prevalence=r['prevalence'], 
                     AUROC=r['AUROC'], 
                     AUPRC=r['AUPRC'],
                     ))
    # the CV folds each have their own positive rate but it's almost negligible
    rows.append(dict(project='KGDR',
                     setting=f"{r['regime']} (10-fold CV)", 
                     what="embedding space diverse",
                     model=f"{r['model']} / {r['kge']}",
                     prevalence=r['prevalence'], 
                     AUROC=r['auroc_mean'], 
                     AUPRC=r['auprc_mean'],
                     ))

cross = pd.DataFrame(rows)
cross['AUPRC_norm'] = norm_auprc(cross['AUPRC'], cross['prevalence'])
cross = cross.sort_values(['project', 'AUPRC_norm'], ascending=[False, False])

table3 = cross.assign( # assign creates new columns in the DataFrame
    **{'positive rate, prevalence': [('-' if pd.isna(v) else f'{v:.3f}') for v in cross['prevalence']],
       'AUROC ': [f'{a:.3f}' + (f' ± {s:.3f}' if pd.notna(s) else '') for a, s in zip(cross['AUROC'], cross['AUROC_std'])],
       'AUPRC ': [f'{a:.3f}' + (f' ± {s:.3f}' if pd.notna(s) else '') for a, s in zip(cross['AUPRC'], cross['AUPRC_std'])],
       'AUPRC norm': cross['AUPRC_norm'].map(lambda v: '-' if pd.isna(v) else f'{v:.3f}')}
)[['project', 'setting', 'what', 'model', 'prevalence', 'AUROC ', 'AUPRC ', 'AUPRC norm']]


display(table3.set_index(['project', 'setting']))


what  \
project   setting                                                   
KGDR      emb_space_diverse               embedding space diverse   
          emb_space_diverse (10-fold CV)  embedding space diverse   
DREAMwalk random / leakage                           random pairs   
          disease_area / leakage            MeSH-category holdout   
          random / no leakage                        random pairs   
          disease_area / no leakage         MeSH-category holdout   

                                                       model  prevalence  \
project   setting                                                          
KGDR      emb_space_diverse               svm_poly / complex    0.091764   
          emb_space_diverse (10-fold CV)  svm_poly / complex    0.091764   
DREAMwalk random / leakage                           XGBoost    0.500000   
          disease_area / leakage                     XGBoost    0.529575   
          random / no leakage                        XGBoost    0.500000   
          disease_area / no leakage                  XGBoost    0.529575   

                                                 AUROC          AUPRC   \
project   setting                                                        
KGDR      emb_space_diverse                       0.983          0.850   
          emb_space_diverse (10-fold CV)          0.918          0.804   
DREAMwalk random / leakage                0.980 ± 0.001  0.979 ± 0.001   
          disease_area / leakage          0.933 ± 0.009  0.935 ± 0.009   
          random / no leakage             0.915 ± 0.003  0.922 ± 0.002   
          disease_area / no leakage       0.663 ± 0.033  0.701 ± 0.028   

                                         AUPRC norm  
project   setting                                    
KGDR      emb_space_diverse                   0.835  
          emb_space_diverse (10-fold CV)      0.784  
DREAMwalk random / leakage                    0.959  
          disease_area / leakage              0.861  
          random / no leakage                 0.844  
          disease_area / no leakage           0.365

## discussion
